Milestone2

In [32]:
import pandas as pd
import numpy as np

In [ ]:
#loading data
data = pd.read_csv("../data/milestone1_output.csv")
data.head()

,id,sender,subject,body,priority,triage_label,clean_text,keywords,triage
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,your invoice of inr is due on please pay to ...,"['invoice', 'inr', 'due', 'please', 'pay', 'av...",notify_human
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,hello team please find the attached weekly rep...,"['hello', 'team', 'please', 'find', 'attached'...",respond_or_act
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,hello team please find the attached weekly rep...,"['hello', 'team', 'please', 'find', 'attached'...",respond_or_act


In [24]:
len(data)

200

In [26]:
#create a column of ideal_response---use if elif logic
#triage _lablerule--return no respone,

def classify_email(text):
    text = text.lower()

    notify_terms = ["refund", "urgent", "password"]
    ignore_terms = ["newsletter", "promotion"]

    if any(word in text for word in notify_terms):
        return "notify_human"

    if any(word in text for word in ignore_terms):
        return "ignore"
    
    return "respond_or_act"



In [27]:
data["predicted_triage"] = data["clean_text"].apply(classify_email)
data[["body", "predicted_triage"]].head()



,body,predicted_triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [28]:
output_cols = ["body", "predicted_triage"]
data.to_csv("../data/milestone1_output.csv", columns=output_cols, index=False)

In [18]:
#select 100 test email

test_data = (
    df.sample(n=100, random_state=42)
        .reset_index(drop=True)
)

test_data.head()


,id,sender,subject,body,priority,triage_label,clean_text,keywords,triage
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore,your order has been shipped and is expected t...,"['order', 'shipped', 'expected', 'deliver']",respond_or_act
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human,hi dont miss our sale with discounts up to on...,"['hi', 'dont', 'miss', 'sale', 'discounts', 's...",ignore
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human,notice your account will be locked unless veri...,"['notice', 'account', 'locked', 'unless', 'ver...",respond_or_act
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond,reminder the client meeting is scheduled at t...,"['reminder', 'client', 'meeting', 'scheduled',...",respond_or_act
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond,your order has been shipped and is expected t...,"['order', 'shipped', 'expected', 'deliver']",respond_or_act


In [29]:
df['clean_text'].head(100)

0     reminder the client meeting is scheduled at  t...
1     your invoice of inr  is due on  please pay to ...
2     reminder the client meeting is scheduled at  t...
3     hello team please find the attached weekly rep...
4     hello team please find the attached weekly rep...
                            ...                        
95    your order  has been shipped and is expected t...
96    security alert multiple failed login attempts ...
97    security alert multiple failed login attempts ...
98    notice your account will be locked unless veri...
99    your order  has been shipped and is expected t...
Name: clean_text, Length: 100, dtype: object

In [30]:
def map_action(triage_type):
    response_map = {
        "ignore": "No action needed",
        "respond": "Send a reply",
        "respond_or_act": "Respond and take required action",
        "notify_human": "Escalate to human"
    }
    return response_map.get(triage_type, "Manual review required")

In [31]:

test_data["ideal_response"] = test_data["triage_label"].apply(map_action)
test_data[["triage_label", "ideal_response"]].head()

,triage_label,ideal_response
0,ignore,No action needed
1,notify_human,Escalate to human
2,notify_human,Escalate to human
3,respond,Send a reply
4,respond,Send a reply


In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")

# Add EMPTY ground-truth columns
df["ideal_intent"] = ""
df["ideal_tone"] = ""

# Save back to same file
df.to_csv("../data/sample_emails_with_triage_200.csv", index=False)

df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,,
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,,
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,,
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,,
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,,


In [2]:
#Create evaluator function (rule-based judge)
def evaluate(agent_output, ideal_action, ideal_tone):
    action_match = agent_output.get("action") == ideal_action
    tone_match = agent_output.get("tone") == ideal_tone
    return int(action_match and tone_match)


In [4]:
#Email Assistant Logic
def email_assistant(email_text):
    text = email_text.lower()

    if "urgent" in text or "deadline" in text or "eod" in text:
        return "notify", "urgent"
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    else:
        return "respond", "neutral"


In [5]:
#Generate Predictions
predictions = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,neutral
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


In [10]:
df.loc[0, ["ideal_intent", "ideal_tone"]] = ["notify", "neutral"]
df.loc[1, ["ideal_intent", "ideal_tone"]] = ["notify", "urgent"]
df.loc[2, ["ideal_intent", "ideal_tone"]] = ["ignore", "polite"]


In [11]:
#Evaluate Accuracy
eval_df = df.merge(pred_df, on="id")
eval_df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,neutral,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,urgent,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,ignore,polite,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,,,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,,,respond,neutral


In [12]:
#Evaluation function (2-point scoring)
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

In [13]:
#Apply evaluation
eval_df["score"] = eval_df.apply(evaluate, axis=1)
eval_df[["body", "ideal_intent", "predicted_intent", "ideal_tone", "predicted_tone", "score"]].head()


,body,ideal_intent,predicted_intent,ideal_tone,predicted_tone,score
0,Reminder: The client meeting is scheduled at 1...,notify,respond,neutral,neutral,1
1,Your invoice of INR 25515.09 is due on 2025-12...,notify,respond,urgent,neutral,0
2,Reminder: The client meeting is scheduled at 1...,ignore,respond,polite,neutral,0
3,"Hello team, please find the attached weekly re...",,respond,,neutral,0
4,"Hello team, please find the attached weekly re...",,respond,,neutral,0


In [14]:
#Compute accuracy (as per instructions)
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy


np.float64(0.25)

In [15]:
eval_df.to_csv(
    "../data/milestone2_output_theertha.csv",
    index=False
)
